# Heart Disease Classifier - Training Walkthrough

This notebook demonstrates the model training pipeline step by step:
1. Load processed data
2. Build preprocessing pipeline (ColumnTransformer)
3. Train Logistic Regression and Random Forest
4. Evaluate with cross-validation
5. Compare results and select best model

In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

from src.data_prep.preprocess import ALL_FEATURE_COLS, TARGET_COL, build_pipeline
from src.evaluation.evaluate import compute_metrics, plot_confusion_matrix, plot_roc_curve

sns.set_theme(style='whitegrid')

## 1. Load Data

In [3]:
df = pd.read_csv('../dataprocessing/processed_cleveland.csv')
X = df[ALL_FEATURE_COLS]
y = df[TARGET_COL]
print(f'Features: {X.shape}, Target: {y.shape}')
print(f'Target distribution: {dict(y.value_counts())}')

Features: (303, 13), Target: (303,)
Target distribution: {0: np.int64(164), 1: np.int64(139)}


## 2. Build Pipeline & Train Models

In [4]:
RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Logistic Regression': build_pipeline(LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
    'Random Forest': build_pipeline(RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=5, random_state=RANDOM_STATE)),
}

results = {}
for name, pipeline in models.items():
    cv_res = cross_validate(pipeline, X, y, cv=cv,
                           scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
                           return_train_score=False)
    results[name] = {metric: np.mean(cv_res[f'test_{metric}']) for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    print(f'\n{name}:')
    for metric, val in results[name].items():
        print(f'  {metric:12s}: {val:.4f} +/- {np.std(cv_res[f"test_{metric}"]):.4f}')


Logistic Regression:
  accuracy    : 0.8580 +/- 0.0476
  precision   : 0.8874 +/- 0.0504
  recall      : 0.7905 +/- 0.0830
  f1          : 0.8343 +/- 0.0600
  roc_auc     : 0.9192 +/- 0.0213

Random Forest:
  accuracy    : 0.8415 +/- 0.0274
  precision   : 0.8716 +/- 0.0386
  recall      : 0.7690 +/- 0.0576
  f1          : 0.8155 +/- 0.0378
  roc_auc     : 0.9186 +/- 0.0188


## 3. Compare Models

In [5]:
comparison = pd.DataFrame(results).T
comparison.plot(kind='bar', figsize=(12, 6), edgecolor='black', linewidth=0.5)
plt.title('Model Comparison (5-Fold Cross-Validation)', fontsize=14, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0.7, 1.0)
plt.legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(comparison.to_string())

                     accuracy  precision    recall        f1   roc_auc
Logistic Regression  0.857978   0.887410  0.790476  0.834301  0.919236
Random Forest        0.841475   0.871554  0.769048  0.815539  0.918621


/var/folders/my/_k1x41pn6f352tn1jv2l2s4w0000gn/T/ipykernel_17484/807314304.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Train Best Model on Full Data & Evaluate

In [6]:
best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f'Best model: {best_name} (ROC-AUC: {results[best_name]["roc_auc"]:.4f})')

best_pipeline = models[best_name]
best_pipeline.fit(X, y)

y_pred = best_pipeline.predict(X)
y_prob = best_pipeline.predict_proba(X)[:, 1]

fig_cm = plot_confusion_matrix(y, y_pred, f'{best_name} - Confusion Matrix')
plt.show()

fig_roc = plot_roc_curve(y, y_prob, f'{best_name} - ROC Curve')
plt.show()

metrics = compute_metrics(y, y_pred, y_prob)
for name, val in metrics.items():
    print(f'{name:12s}: {val:.4f}')

Best model: Logistic Regression (ROC-AUC: 0.9192)
accuracy    : 0.8713
precision   : 0.8968
recall      : 0.8129
f1          : 0.8528
roc_auc     : 0.9390


/var/folders/my/_k1x41pn6f352tn1jv2l2s4w0000gn/T/ipykernel_17484/568791492.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/my/_k1x41pn6f352tn1jv2l2s4w0000gn/T/ipykernel_17484/568791492.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
